# Pseudoreplication and Independence

## Overview
Pseudoreplication (Hurlbert 1984) occurs when statistical analyses treat non-independent observations as independent replicates, inflating the effective sample size and producing spuriously small p-values.

**Hurlbert's taxonomy of pseudoreplication:**

| Type | Description | Example |
|---|---|---|
| **Simple** | Subsamples within a single experimental unit treated as replicates | Multiple fish from one tank as independent replicates |
| **Temporal** | Repeated measurements on the same unit treated as independent | Weekly measurements of one plot as separate observations |
| **Sacrificial** | Multiple experimental units per treatment but only treatment means analysed | Averaging across replicates before testing |
| **Implicit** | Non-independent units due to spatial or temporal proximity | Adjacent plots with shared nutrient run-off |

**The core issue:** the denominator of your test statistic should reflect the number of truly independent experimental units (n = replicates), not the total number of observations made.

**Reference:** Hurlbert (1984) Ecological Monographs 54:187–211; Underwood (1997) ch. 9; Quinn & Keough (2002) ch. 7

---

In [ ]:
library(tidyverse); library(lme4); library(lmerTest)
set.seed(55)
# Scenario: effect of predator addition on invertebrate abundance
# Design: 6 tanks (3 control, 3 predator); 8 core samples per tank
# TRUE replication: n=3 per treatment
n_tanks <- 6
n_cores <- 8
tank_id   <- rep(1:n_tanks, each = n_cores)
treatment <- rep(c("control", "predator", "control", "predator",
                   "control", "predator"), each = n_cores)
# True treatment effect: predators reduce abundance by ~30%
tank_re <- rep(rnorm(n_tanks, 0, 1.2), each = n_cores)  # tank-level variation
base_log_mu <- ifelse(treatment == "control", 3.5, 3.0)
mu <- exp(base_log_mu + tank_re)
abundance <- rpois(n_tanks * n_cores, lambda = mu)

dat <- data.frame(tank_id   = factor(tank_id),
                  treatment = factor(treatment),
                  core      = 1:n_cores,
                  abundance = abundance)

cat("Dataset: n =", nrow(dat), "core samples across",
    n_tanks, "tanks (3 per treatment)\n")
cat("True replication per treatment: n = 3 tanks\n")
dat |> group_by(treatment, tank_id) |>
  summarise(mean_abund = round(mean(abundance), 1), .groups = "drop") |> print()

---
## Demonstrating the problem: pseudoreplicated vs correct analysis

In [ ]:
# WRONG: treating each core sample as an independent replicate
t_wrong <- t.test(abundance ~ treatment, data = dat)
cat("=== PSEUDOREPLICATED analysis (n=48 cores treated as replicates) ===\n")
cat(sprintf("t = %.3f, df = %.1f, p = %.5f\n",
            t_wrong$statistic, t_wrong$parameter, t_wrong$p.value))
cat("n per group = 24 — INFLATED. Only 3 independent tanks per treatment!\n")

# ALSO WRONG: tank means (sacrificial pseudoreplication)
tank_means <- dat |>
  group_by(tank_id, treatment) |>
  summarise(mean_abund = mean(abundance), .groups = "drop")
t_means <- t.test(mean_abund ~ treatment, data = tank_means)
cat("\n=== TANK MEANS t-test (correct n but ignores within-tank structure) ===\n")
cat(sprintf("t = %.3f, df = %.1f, p = %.4f\n",
            t_means$statistic, t_means$parameter, t_means$p.value))
cat("n per group = 3. Correct replication but discards within-tank information.\n")

In [ ]:
# CORRECT approach 1: Linear mixed model (tank as random effect)
# Uses all core-level data but accounts for non-independence within tanks
m_lmm <- lmer(log(abundance + 1) ~ treatment + (1 | tank_id), data = dat)
cat("=== CORRECT: Linear Mixed Model (random tank effect) ===\n")
print(summary(m_lmm)$coefficients)
cat("\nRandom effects:\n")
print(VarCorr(m_lmm))
cat("\nIntraclass correlation (ICC) = proportion of variance between tanks:\n")
vc <- as.data.frame(VarCorr(m_lmm))
icc <- vc$vcov[1] / sum(vc$vcov)
cat(sprintf("  ICC = %.3f (%.0f%% of variance is between-tank)\n", icc, icc*100))
cat("High ICC confirms that tank is a real clustering factor — pseudoreplication matters here.\n")

In [ ]:
# Design audit checklist
cat("\n=== DESIGN AUDIT CHECKLIST ===\n")
checklist <- c(
  "1. Is the treatment applied to the experimental unit, or to subsamples of it?",
  "2. Are replicates spatially or temporally interspersed across treatments?",
  "   (contiguous plots of the same treatment risk spatial autocorrelation)",
  "3. Are repeated measurements on the same unit included?",
  "   (use mixed models or repeated measures ANOVA)",
  "4. Are observations from the same cluster (tank, plot, individual) independent?",
  "   (compute ICC; if high, clustering matters)",
  "5. Is the n reported = experimental units, not total observations?",
  "6. Is randomisation documented (which units got which treatment)?"
)
cat(paste(checklist, collapse = "\n"), "\n")

# Detecting pseudoreplication: intraclass correlation
cat("\n--- ICC across multiple grouping levels ---\n")
library(performance)
cat("icc(m_lmm) computes variance partitioning:\n")
print(icc(m_lmm))

---
## Common Pitfalls

**1. Confusing the experimental unit with the observational unit**
The experimental unit is the entity to which a treatment is randomly assigned. All measurements taken from within one experimental unit are subsamples, not replicates. The t-test, ANOVA, and most tests require independence at the level of the experimental unit.

**2. Assuming that averaging across subsamples fixes pseudoreplication**
Averaging cores within a tank (sacrificial pseudoreplication) gives you the correct n but discards valid within-unit information and ignores the variance structure. Linear mixed models use all observations while correctly partitioning variance.

**3. Spatial contiguity as a hidden form of pseudoreplication**
Replicates assigned to the same treatment in spatially contiguous blocks are not fully independent — they may share environmental gradients, resources, or disturbance history. Intersperse replicates of different treatments throughout the study area (Hurlbert's interspersion criterion).

**4. Temporal pseudoreplication in before-after designs**
Measurements taken on the same unit at multiple time points are not independent. Treat time as a within-subject factor in repeated measures ANOVA or mixed models; do not pool all time points as independent replicates.

**5. Reporting inflated degrees of freedom**
A tell-tale sign of pseudoreplication is very large df in published ANOVA tables (e.g., df_error = 200 in a study with 6 experimental units). The df for testing treatment effects should be based on the number of experimental units, not measurements.

**6. Dismissing pseudoreplication in observational studies**
Pseudoreplication applies equally to observational studies. Sampling multiple fish from the same individual or multiple plots from the same farm introduces hierarchical non-independence that must be accounted for, even without experimental manipulation.


---
*r_methods_library - Samantha McGarrigle*